# Sinh hình cho bài GRACE bằng WSL

Mở notebook trong VS Code kết nối **WSL**, chọn kernel Python từ `.venv-wsl` của repo, rồi chạy **Restart Kernel and Run All Cells**. Nếu môi trường chưa có thư viện vẽ, chạy `.venv-wsl/bin/python -m pip install matplotlib` từ thư mục gốc repo trong WSL.

Notebook dùng đúng ZIP của Option 1 ngày 11/11 và hồ sơ đầu vào của báo cáo PA1. Đầu ra là hai hình ROC và hai hình điểm hoạt động cho Devign và ReVeal, đặt trong thư mục cha `figures`. Các cell chạy trên điểm đã lưu, dùng CPU.

In [ ]:
from hashlib import sha256
from pathlib import Path
from tempfile import TemporaryDirectory
from zipfile import ZipFile
import importlib.util
import json
import subprocess
import sys

PROJECT = Path("/mnt/d/JUNIOR/LabThayTho/GRACE-summary")
if sys.platform == "win32":
    PROJECT = Path("D:/JUNIOR/LabThayTho/GRACE-summary")

FIGURES = PROJECT / "[Official]-Paper" / "figures"
ZIP = FIGURES / "generating-figures" / "grace_diag_kit.zip"
PROVENANCE = PROJECT / "Runs-management/analysis_pa1_20261111/full_tables/provenance.json"
KIT_SHA256 = "2b29dfd3a369c7defecc506af2db9aa18d0c3ce8fdb6970b47734bf471fb2a2b"
PROVENANCE_SHA256 = "500d56d07124716919fb90949fc3bdb7cdef1146701c715fb9304f29d54ef647"

def digest(path):
    with path.open("rb") as source:
        return sha256(source.read()).hexdigest()

def verify_hash(path, expected):
    actual = digest(path)
    if actual != expected:
        raise ValueError(f"SHA-256 không khớp: {path}\n{actual} != {expected}")

verify_hash(ZIP, KIT_SHA256)
verify_hash(PROVENANCE, PROVENANCE_SHA256)

missing = [name for name in ("numpy", "scipy", "sklearn", "matplotlib")
           if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError(f"Kernel {sys.executable} thiếu: {', '.join(missing)}")

import matplotlib
import numpy as np
from PIL import Image

print("Kernel:", sys.executable)
print("Kit SHA-256:", KIT_SHA256)

## 1. Chọn và kiểm tra dữ liệu đầu vào

Dùng seed 42 và chín arm trong đầu vào PA1: B0, B1, B1-shuf, B2, B2-flip, B2-rep, B2-rand, B2-nolabel và B2-labelonly. B3 và B1-H2 là phân tích thăm dò theo lộ trình Option 1. Đường ROC dùng B0, B2 và SUP-LR. Hình điểm hoạt động dùng chín arm B, theo cách hàm gốc lọc các arm bắt đầu bằng B.

In [ ]:
import csv

inputs = json.loads(PROVENANCE.read_text(encoding="utf-8"))["inputs_sha256"]
dataset_sizes = {"devign": 2732, "reveal": 2274}
expected_arms = {
    "B0", "B1", "B1-shuf", "B2", "B2-flip", "B2-rep",
    "B2-rand", "B2-nolabel", "B2-labelonly",
}
excluded_paths = ("/b1h2_test_latest/", "/b3_verified_patch_test_10h/")
sources = [
    PROJECT / name for name in inputs
    if name.endswith("/test/records.jsonl")
    and not any(segment in name for segment in excluded_paths)
]

manifest_base = PROJECT / "01_Author_Baseline_Unified/output/research_freeze_20261101"
anchor_base = PROJECT / "STEP6_supervised_anchors/runs/anchor/locked_step6_20261108"
test_labels = {}
anchor_files = {}

def read_rows(path):
    with path.open(encoding="utf-8") as stream:
        return [json.loads(line) for line in stream if line.strip()]

def verify_input(path):
    verify_hash(path, inputs[path.relative_to(PROJECT).as_posix()])

for dataset, size in dataset_sizes.items():
    manifest = manifest_base / dataset / "test_manifest.jsonl"
    anchor_files[dataset] = anchor_base / dataset / "anchor_scores.csv"
    verify_input(manifest)
    verify_input(anchor_files[dataset])
    rows = read_rows(manifest)
    test_labels[dataset] = {str(row["id"]): row["label"] for row in rows}
    if len(rows) != size or len(test_labels[dataset]) != size:
        raise ValueError(f"{dataset}: số ID trong manifest khác {size}")

for source in sources:
    verify_input(source)

if len(sources) != len(dataset_sizes) * len(expected_arms):
    raise ValueError(f"Số file test nguồn khác dự kiến: {len(sources)}")
print(f"Đã kiểm SHA-256 của {len(sources)} records, 2 manifest và 2 file anchor.")

## 2. Giải nén kit nguyên bản và gom records

Kit được giải nén vào thư mục tạm. Notebook chạy importer gốc hai lần để kiểm tra trước rồi mới ghi records đã gom. Notebook gọi Python với `-X utf8` để log tiếng Việt hiển thị đúng cả khi chạy từ Windows. Mọi kiểm tra thêm nằm trong notebook.

In [ ]:
workspace = TemporaryDirectory(prefix="grace-pa1-figures-")
scratch = Path(workspace.name)

with ZipFile(ZIP) as archive:
    archive.extractall(scratch)

kit_dir = scratch / "grace_diag_kit"
runs = scratch / "records"
command = [
    sys.executable, "-X", "utf8", str(kit_dir / "scripts/import_records.py"),
    "--src", *map(str, sources), "--dst", str(runs),
]
subprocess.run([*command, "--dry-run"], check=True)
subprocess.run(command, check=True)

sys.path.insert(0, str(kit_dir / "src"))
from grace_diag.evaluate_final import figures, load_anchor, load_runs

if Path(sys.modules[figures.__module__].__file__).resolve() != (
    kit_dir / "src/grace_diag/evaluate_final.py"
).resolve():
    raise RuntimeError("Kernel đang dùng kit khác. Chọn Restart Kernel and Run All Cells.")

data = load_runs(runs)
expected_keys = {
    (dataset, arm, 42, "test") for dataset in dataset_sizes for arm in expected_arms
}
if set(data) != expected_keys:
    raise ValueError(f"Nhóm records không khớp: {set(data) ^ expected_keys}")

for (dataset, arm, seed, split), records in sorted(data.items()):
    labels = test_labels[dataset]
    if records.keys() != labels.keys():
        raise ValueError(f"ID test không khớp manifest: {dataset}/{arm}/{seed}")
    for row_id, row in records.items():
        if row["label"] != labels[row_id]:
            raise ValueError(f"Nhãn không khớp manifest: {dataset}/{arm}/{row_id}")
        if row.get("s_bounded"):
            raise ValueError(f"Phát hiện điểm bị chặn ngoài phạm vi đã đối soát: {dataset}/{arm}/{row_id}")
        if row["valid"] and not (
            np.isfinite(row["s"]) and 0 <= row["s"] <= 1
            and row["pred_argmax"] == int(row["s"] >= 0.5)
        ):
            raise ValueError(f"Điểm hoặc dự đoán tại 0,5 không khớp: {dataset}/{arm}/{row_id}")
print(f"Đã đối chiếu ID, nhãn và ngưỡng 0,5 của {len(data)} nhóm test.")

## 3. Nạp SUP-LR và vẽ bằng hàm gốc

Phụ lục B gọi `evaluate_final` để sinh cả bảng lẫn hình. Notebook này gọi trực tiếp `figures()` của chính module đó vì chỉ sinh hình. Hàm ROC dùng điểm liên tục, còn hình điểm hoạt động dùng `pred_argmax`, đã được kiểm tra tương ứng với `s ≥ 0,5`. Vì vậy notebook truyền `{}` cho tham số `taus` mà hàm vẽ không sử dụng. Khi B2 và B2-rep có cùng tọa độ, notebook gộp hai tên thành một nhãn ngay trước lúc lưu PNG để chữ không đè lên nhau. Tọa độ và dữ liệu vẫn do hàm gốc tính. B2-rand trên ReVeal giữ cảnh báo hai runtime của báo cáo PA1. Việc sinh bảng theo ngưỡng khóa hoặc ngưỡng hiệu chỉnh vẫn thuộc lượt đánh giá đầy đủ của Phụ lục B.

In [ ]:
labels_by_split = {(dataset, "test"): labels for dataset, labels in test_labels.items()}
for dataset, anchor_file in anchor_files.items():
    with anchor_file.open(encoding="utf-8", newline="") as source:
        anchor_ids = [
            row["id"] for row in csv.DictReader(source)
            if (row["dataset"], row["arm"], row["split"], row["seed"])
            == (dataset, "SUP-LR", "test", "42")
        ]
    if len(anchor_ids) != dataset_sizes[dataset] or set(anchor_ids) != set(test_labels[dataset]):
        raise ValueError(f"ID hoặc số dòng SUP-LR không khớp manifest: {dataset}")
    load_anchor(anchor_file, labels_by_split, data)
    anchor = data[(dataset, "SUP-LR", 42, "test")]
    if anchor.keys() != test_labels[dataset].keys():
        raise ValueError(f"Thiếu điểm SUP-LR sau khi nạp: {dataset}")
    if not all(np.isfinite(row["s"]) and 0 <= row["s"] <= 1 for row in anchor.values()):
        raise ValueError(f"Điểm SUP-LR ngoài miền [0, 1]: {dataset}")

staged = scratch / "plots"
staged.mkdir()
from matplotlib.figure import Figure
from unittest.mock import patch

save_figure = Figure.savefig

def save_with_readable_labels(figure, filename, *args, **kwargs):
    axis = figure.axes[0]
    axis.set_title(axis.get_title().replace("devign", "Devign").replace("reveal", "ReVeal"))
    if Path(filename).name.startswith("fig_operating_points_"):
        axis.set_xlabel("Positive prediction rate (s ≥ 0.5)")
        labels_by_point = {}
        for label in figure.axes[0].texts:
            labels_by_point.setdefault(label.xy, []).append(label)
        for labels in labels_by_point.values():
            if len(labels) < 2:
                continue
            first, *overlapping = labels
            first.set_text(" / ".join(label.get_text() for label in labels))
            first.set_position((-4, 4))
            first.set_horizontalalignment("right")
            for label in overlapping:
                label.remove()
        figure.tight_layout()
    return save_figure(figure, filename, *args, **kwargs)

with patch.object(Figure, "savefig", save_with_readable_labels):
    figures(data, {}, staged, arms=("B0", "B2", "SUP-LR"), seed=42)

## 4. Kiểm tra và chép bốn PNG vào thư mục cha

Chỉ chép kết quả khi cả bốn ảnh đã sinh và đọc được. Khi chạy lại toàn bộ notebook, bốn PNG được cập nhật từ cùng bộ đầu vào.

In [ ]:
import shutil

expected_pngs = {
    f"fig_{kind}_{dataset}.png"
    for kind in ("roc", "operating_points") for dataset in dataset_sizes
}
actual_pngs = {path.name for path in staged.glob("*.png")}
if actual_pngs != expected_pngs:
    raise RuntimeError(f"Danh sách PNG không khớp: {actual_pngs ^ expected_pngs}")

for name in sorted(expected_pngs):
    source = staged / name
    with Image.open(source) as image:
        image.verify()
    with Image.open(source) as image:
        print(f"{name}: {image.width} × {image.height}, SHA-256 {digest(source)}")

verify_hash(ZIP, KIT_SHA256)
for name in sorted(expected_pngs):
    shutil.copyfile(staged / name, FIGURES / name)

workspace.cleanup()
print("Đã lưu 4 hình tại:", FIGURES)